# Supply Orders: Loading and Typing Messy Mock Data

This notebook takes a raw CSV export where **every value is text** and turns it into a properly typed pandas DataFrame. It's the first step of almost any data pipeline: until dates are real dates and prices are real numbers, you can't sort, filter, total, or trust anything.

**What it does, in order**

1. Loads local mock data (`mock_supply_orders.csv`) into a DataFrame, keeping everything as raw text
2. Prints a range of rows so you can see what you're working with
3. Strips leading and trailing whitespace from headers and values
4. Converts date text to a **date** type
5. Converts time text to a **time** type
6. Converts quantity text to **integers**
7. Converts price text to **exact decimals**
8. Converts single-character **Y/N and T/F flags** to booleans
9. Reports every value that couldn't be converted, instead of silently dropping it

| Column | Raw text example | Converted type |
|---|---|---|
| `order_date` | `" 08/21/2026 "` | `date32` |
| `order_time` | `"  02:12 PM"` | `time64[us]` |
| `quantity` | `"  1,200   "` | `int64` |
| `unit_price` | `"$ 1,249.50"` | `decimal128(12, 2)` |
| `rush_order` | `" y "` | `boolean` |
| `received` | `"T"` | `boolean` |

**Requirements:** `pip install "pandas>=2.1" "pyarrow>=14" jupyterlab`

> The mock file includes a handful of deliberately bad values (an impossible date, a blank time, `TBD` as a quantity, `call vendor` as a price, a fractional quantity, a missing date, a `?` flag). They're there so the validation step at the end has something to catch.

## 1. Setup and configuration

Everything you'd reasonably want to change lives in this cell: the file path, which rows to print, and the text formats the source system uses. Declaring formats explicitly is faster than letting pandas guess, and it removes ambiguity (is `03/04/2026` March 4th or April 3rd? With an explicit format, there's no question).

In [ ]:
from datetime import time
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
from pathlib import Path

import pandas as pd
import pyarrow as pa

# ---- Configuration ----------------------------------------------------------
DATA_PATH = Path("mock_supply_orders.csv")  # local mock data, same folder as this notebook
START_ROW, END_ROW = 5, 15                  # rows to print (end is exclusive, like Python slicing)

DATE_FORMAT = "%m/%d/%Y"                    # e.g. 08/21/2026
TIME_FORMAT = "%I:%M %p"                    # e.g. 02:12 PM
NULL_TOKENS = ["", "N/A", "NA", "NULL", "null", "-"]  # text that really means "no value"
TRUE_FLAGS = {"Y", "T"}                     # single-character yes values (compared after uppercasing)
FALSE_FLAGS = {"N", "F"}                    # single-character no values

PRICE_SCALE = Decimal("0.01")               # round prices to cents
PRICE_TYPE = pa.decimal128(12, 2)           # 12 total digits, 2 after the decimal point

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
print(f"pandas {pd.__version__} | pyarrow {pa.__version__}")

## 2. Load the local mock data

Two settings matter here:

- `dtype=str` tells pandas to read **every** column as text. Without it, pandas tries to be helpful and guesses types, and guesses on messy data go wrong quietly (a quantity column with one `TBD` in it becomes entirely text anyway, and a zip code like `07701` loses its leading zero).
- `keep_default_na=False, na_filter=False` stops pandas from deciding on its own what counts as "missing." We'll make that call ourselves after cleaning.

The result is the file exactly as it sits on disk, so every change after this point is deliberate and visible.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Mock data not found at {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False, na_filter=False)

print(f"Loaded {len(df):,} rows x {df.shape[1]} columns from {DATA_PATH.name}\n")
print(df.dtypes)

Notice the column names in that output have stray spaces (`" facility "`, `" quantity "`). That's common in exports from older systems and will break code like `df["facility"]` until it's fixed.

## 3. Print a range of the data

`iloc` selects rows by position. Change `START_ROW` / `END_ROW` in the setup cell to look at a different slice.

In [ ]:
print(f"Rows {START_ROW} through {END_ROW - 1} (raw):\n")
print(df.iloc[START_ROW:END_ROW].to_string())

Spaces are invisible in a normal printout, so here are the same first few rows shown with `repr()`, which wraps each value in quotes and displays tabs as `\t`. This is what the conversion steps are actually up against:

In [ ]:
print(df.iloc[START_ROW:START_ROW + 4].map(repr).to_string())

In [ ]:
def whitespace_report(frame: pd.DataFrame) -> pd.DataFrame:
    """Per column: is the header padded, and how many cells have leading/trailing whitespace?"""
    rows = {}
    for col in frame.columns:
        values = frame[col].fillna("")
        rows[repr(col)] = {
            "header_padded": col != col.strip(),
            "padded_cells": int((values != values.str.strip()).sum()),
        }
    return pd.DataFrame.from_dict(rows, orient="index")

whitespace_report(df)

## 4. Strip leading and trailing whitespace

Three things happen here:

1. **Headers** get trimmed so columns can be referenced by their real names.
2. **Every value** gets trimmed. `str.strip()` removes spaces, tabs, and line breaks from both ends but leaves internal spaces alone (`"Mop Head, Looped"` stays intact).
3. **Placeholder text becomes a true missing value.** After trimming, a cell that was just spaces is now an empty string. An empty string isn't a date or a number, so if we left it as-is it would show up later as a conversion failure. Marking it as missing (`NA`) lets us tell the difference between *"the source had no value"* and *"the source had a value we couldn't read."*

We also save a copy of the cleaned text (`raw`) before any type conversion. The validation step uses it to show exactly which original values failed.

In [ ]:
df.columns = df.columns.str.strip()
df = df.apply(lambda col: col.str.strip())
df = df.replace(NULL_TOKENS, pd.NA)

raw = df.copy()  # cleaned text, kept for the validation report

whitespace_report(df)

Every count is now zero.

## 5. Convert date text to a date type

`pd.to_datetime` parses the text using the declared format. `errors="coerce"` means a value that can't be parsed (like `13/45/2026`) becomes missing instead of crashing the whole notebook. Nothing is lost silently, though: section 9 reports it.

pandas' default for dates is a full timestamp (`datetime64`), which carries a time-of-day of midnight. Since this column is a date and nothing more, we store it as Arrow's `date32` type. It's a true calendar date, it's half the memory, and it won't produce confusing `00:00:00` values when joined or exported.

In [ ]:
parsed_dates = pd.to_datetime(df["order_date"], format=DATE_FORMAT, errors="coerce")
df["order_date"] = parsed_dates.astype(pd.ArrowDtype(pa.date32()))

print(df["order_date"].dtype)
print(df["order_date"].head())

## 6. Convert time text to a time type

pandas doesn't have a native time-of-day type. The standard workaround is:

1. Parse with `to_datetime`, which attaches a placeholder date (`1900-01-01`)
2. Use `.dt.time` to throw the placeholder date away
3. Store the result as Arrow `time64`, so it's a real typed column instead of a column of generic Python objects

The 12-hour `AM/PM` text is normalized to 24-hour time in the process (`02:12 PM` becomes `14:12:00`).

In [ ]:
parsed_times = pd.to_datetime(df["order_time"], format=TIME_FORMAT, errors="coerce")
df["order_time"] = parsed_times.dt.time.astype(pd.ArrowDtype(pa.time64("us")))

print(df["order_time"].dtype)
print(df["order_time"].head())

## 7. Convert quantity text to integers

The obvious one-liner, `df["quantity"].astype(int)`, fails in two ways on real data: it crashes on `"1,200"` and blanks, and if you work around that by converting to float first, `2.5` gets **silently truncated to 2**. That's an inventory error nobody would ever notice.

So this cell:

1. Removes thousands separators (`1,200` becomes `1200`)
2. Converts to a number, turning non-numeric text like `TBD` into missing
3. **Rejects fractional values** instead of truncating them, and says so
4. Stores the result as Arrow `int64`, which (unlike NumPy's `int64`) can hold missing values without turning the whole column into floats

In [ ]:
qty_numeric = pd.to_numeric(df["quantity"].str.replace(",", "", regex=False), errors="coerce")

fractional = qty_numeric.notna() & (qty_numeric % 1 != 0)
if fractional.any():
    print(f"Rejected {fractional.sum()} fractional quantity value(s): "
          f"{df.loc[fractional, 'quantity'].tolist()}\n")

df["quantity"] = qty_numeric.mask(fractional).astype(pd.ArrowDtype(pa.int64()))

print(df["quantity"].dtype)
print(df["quantity"].head())

## 8. Convert price text to exact decimals

Money should never be stored as a float. Floats are binary approximations, which is why in Python `0.1 + 0.2` equals `0.30000000000000004`. Across thousands of line items those rounding errors add up to totals that don't reconcile with the finance system.

Python's `Decimal` type does exact base-10 arithmetic, the same way an accountant would. This cell:

1. Strips currency formatting (`$`, commas, the space in `$ 1,249.50`)
2. Parses the result as a `Decimal`, returning missing for text like `call vendor`
3. Rounds to cents using standard half-up rounding (`64.0` becomes `64.00`)
4. Stores the column as Arrow `decimal128(12, 2)`: exact, typed, and fast, instead of a column of loose Python objects

In [ ]:
def parse_price(value):
    """'$ 1,249.50' -> Decimal('1249.50'). Anything unparseable -> None."""
    if pd.isna(value):
        return None
    cleaned = value.replace("$", "").replace(",", "").replace(" ", "")
    try:
        amount = Decimal(cleaned)
    except InvalidOperation:
        return None
    if not amount.is_finite():  # rejects text like "NaN" or "Infinity"
        return None
    return amount.quantize(PRICE_SCALE, rounding=ROUND_HALF_UP)

df["unit_price"] = df["unit_price"].map(parse_price).astype(pd.ArrowDtype(PRICE_TYPE))

print(df["unit_price"].dtype)
print(df["unit_price"].head())

## 9. Convert Y/N and T/F flags to booleans

Source systems usually store yes/no as a single character, and they don't agree on which: `rush_order` uses **Y/N**, `received` uses **T/F**. Three things to watch:

- **Map explicitly, in any case.** `y` has to count the same as `Y`, so values are uppercased before matching.
- **Never use `astype(bool)`.** Python treats any non-empty text as True, so `"N"` and `"F"` both become True, with no error and no warning. The cell below shows it.
- **Unknown stays unknown.** Anything that isn't a known yes or no (like `?`) becomes `<NA>` and shows up in the validation report, instead of quietly turning into False.

In [ ]:
FLAG_COLS = ["rush_order", "received"]

for col in FLAG_COLS:
    print(f"{col} raw values: {df[col].value_counts(dropna=False).to_dict()}")
trap = df["rush_order"].dropna().astype(bool)
print(f"\nThe astype(bool) trap: {trap.sum()} of {len(trap)} rush_order values become True, every 'N' included\n")

for col in FLAG_COLS:
    upper = df[col].str.upper()
    flags = pd.Series(pd.NA, index=df.index, dtype="boolean")
    flags[upper.isin(TRUE_FLAGS)] = True
    flags[upper.isin(FALSE_FLAGS)] = False
    df[col] = flags

print(df[FLAG_COLS].dtypes.to_string())
print(df[FLAG_COLS].apply(lambda s: s.value_counts(dropna=False)).to_string())

## 10. Validate: what didn't convert?

`errors="coerce"` is only safe if you check what it coerced. This compares the cleaned text (`raw`) against the converted data and separates two very different situations:

- **missing_in_source**: the file had no value. That's a data completeness issue for the source system owner.
- **failed_conversion**: the file had a value, but it wasn't valid. That's a data quality issue, and each one is listed below with the original text so someone can fix it at the source.

In a production pipeline, the exceptions table is what you'd route to a quarantine table or a ticket rather than just printing it.

In [ ]:
CONVERTED_COLS = ["order_date", "order_time", "quantity", "unit_price", "rush_order", "received"]

failed_mask = raw[CONVERTED_COLS].notna() & df[CONVERTED_COLS].isna()

summary = pd.DataFrame({
    "missing_in_source": raw[CONVERTED_COLS].isna().sum(),
    "failed_conversion": failed_mask.sum(),
    "converted_ok": df[CONVERTED_COLS].notna().sum(),
})
print("Conversion summary:\n")
print(summary.to_string())

stacked = failed_mask.stack()
failed_cells = stacked[stacked].index  # (row, column) pairs
exceptions = pd.DataFrame({
    "row": failed_cells.get_level_values(0),
    "order_id": df.loc[failed_cells.get_level_values(0), "order_id"].to_numpy(),
    "column": failed_cells.get_level_values(1),
    "raw_value": [raw.at[row, col] for row, col in failed_cells],
})
print(f"\n{len(exceptions)} value(s) failed conversion:\n")
print(exceptions.to_string(index=False) if len(exceptions) else "None")

## 11. Final result

The same row range as section 3, now clean and typed. Rows with a problem show `<NA>` in the affected column, and every one of those is accounted for in the report above.

In [ ]:
print(df.dtypes, "\n")
print(f"Rows {START_ROW} through {END_ROW - 1} (cleaned and typed):\n")
print(df.iloc[START_ROW:END_ROW].to_string())

### Proof the types actually work

Operations that were impossible on the raw text now just work: exact money math, time-of-day filtering, and grouping by calendar month.

In [ ]:
checks = df.assign(line_total=df["quantity"] * df["unit_price"])  # exact Decimal math

print(f"Total spend:         ${checks['line_total'].sum():,}")
print("Orders before noon:  ", int((checks["order_time"] < time(12, 0)).sum()))
print("Earliest order date: ", checks["order_date"].min())
print("Rush orders not yet received:", int((checks["rush_order"] & ~checks["received"]).sum()))
print("\nSpend by month:")
print(checks.groupby(checks["order_date"].dt.month)["line_total"].sum().to_string())